### Merging | Joining | Concatenating

In [ ]:
import numpy as np
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
# Datasets
courses = pd.read_csv(os.getenv("COURSES_FILE_PATH"))
ipl = pd.read_csv(os.getenv("IPL_BALL_DELIVERIES_FILE_PATH"))
matches = pd.read_csv(os.getenv("MATCHES_FILE_PATH"))
register_month1 = pd.read_csv(os.getenv("REGISTER_MONTH_1_FILE_PATH"))
register_month2 = pd.read_csv(os.getenv("REGISTER_MONTH_2_FILE_PATH"))
students = pd.read_csv(os.getenv("STUDENT_FILE_PATH"))

In [ ]:
# concat
# pd.concat([register_month1,register_month2]) #* vertical concat, index retained
all_reg = pd.concat([register_month1,register_month2],ignore_index=True)

In [ ]:
# multi-index Data
reg = pd.concat([register_month1,register_month2],keys=["November", "December"])
pd.concat([register_month1,register_month2],keys=["November", "December"]).loc[("November",8)]
reg.shape

In [ ]:
# Horizontal concat
pd.concat([register_month1,register_month2],keys=["November", "December"],axis=1)

##### join

In [ ]:
all_reg
# students
df = pd.DataFrame({
    "student_id":[101, 102, 103],
    "name":["Alpha", "Beta", "Gamma"],
    "partner":[3,5,8]
})

students = pd.concat([students,df],ignore_index=True)
students

##### inner join
Returns only the rows where the key exists in both DataFrames.

In [ ]:
# inner join
students.merge(all_reg,how="inner",on="student_id")
# students only present in all_reg not printed

##### left join
Returns all rows from the left DataFrame and matching rows from the right DataFrame. If no match exists, missing values become NaN

In [ ]:
# left join
students.merge(all_reg,how="left",on="student_id")
# course with no student registration printed

##### right join
Returns all rows from the right DataFrame and matching rows from the left DataFrame.

In [ ]:
students.merge(all_reg,how="right",on="student_id")
# left - students
# right - all_reg

In [ ]:
students.merge(all_reg,how="left",on="student_id")

##### outer join
Returns all rows from both DataFrames. Where no matching row exists, missing values are filled with NaN

In [ ]:
students.merge(all_reg,how="outer",on="student_id")

In [ ]:
# total revenue
total_revenue = courses.merge(all_reg, how="inner", on="course_id")["price"].sum()
print("Total Revenue: ", total_revenue)

In [ ]:
# revenue in each month
monthly_data = pd.concat([register_month1,register_month2],keys=["November", "December"]).reset_index().merge(courses,on="course_id").groupby("level_0").sum()["price"]

monthly_data

In [ ]:
# name course price
all_reg.merge(students, on="student_id").merge(courses, on="course_id")[["name","course_name","price"]]

In [ ]:
# bar chart for revenue/course
courses.merge(all_reg, on="course_id").groupby("course_name").sum().plot(kind="bar", figsize=(8,4), color="#e04526", edgecolor='none', alpha=1)

In [ ]:
# loyal learners
common_studentid = np.intersect1d(register_month1["student_id"],register_month2["student_id"])
students[students["student_id"].isin(common_studentid)][["name","student_id"]]

In [ ]:
# no registration (course)
no_reg = (np.setdiff1d(courses["course_id"],all_reg['course_id']))
courses[courses["course_id"].isin(no_reg)]

In [ ]:
# no registration (student)
students[students["student_id"].isin(np.setdiff1d(students["student_id"],all_reg["student_id"]))]["name"]

##### self join
merging a DataFrame with itself

In [ ]:
students.merge(students,left_on="student_id",right_on="partner")[["name_x","name_y"]]

In [ ]:
# students.merge(all_reg,on="student_id")["name"].value_counts().sort_values(ascending=False) 
#! incorrect approach, there might be 2 student with same name; aplha(3), alpha(2) -> alpha(5)

learners = pd.DataFrame({
    "student_id": [101, 102, 103, 104, 105],
    "name": ["Aplha", "Beta", "Gamma", "Gamma", "Delta"],
    "branch": ["CSE", "IT", "ECE", "AI", "CSE"]
})

# Course registrations
enroll = pd.DataFrame({
    "student_id": [101, 101, 102, 103, 101, 104, 104, 105, 104, 106],
    "course": [
        "Python",
        "DBMS",
        "Python",
        "ML",
        "DSA",
        "DBMS",
        "OS",
        "ML",
        "Python",
        "Java"
    ]
})

learners.merge(enroll,on="student_id")["name"].value_counts().sort_values(ascending=False)
# Gamma(id:103) enrolled in 1 course
# Gamma(id:104) enrolled in 3 courses
# output: Gamma enrolled in 4 courses

learners.merge(enroll,on="student_id").groupby(["name", "student_id"])["name"].count().sort_values(ascending=False) 


In [ ]:
students.merge(all_reg,on="student_id").groupby(["name", "student_id"])["name"].count().sort_values(ascending=False) 


In [ ]:
# Profitable
students.merge(all_reg,on="student_id").merge(courses,on="course_id")[["name", "price"]].groupby("name").sum().sort_values(by="price",ascending=False)


##### Merge on IPL Data

In [ ]:
# six/match ratio
venue_name = ipl.merge(matches,left_on="match_id", right_on="id")
sixes = venue_name[venue_name["batsman_runs"] == 6].groupby("venue")["batsman_runs"].count()
matches_stadium = matches["venue"].value_counts()

sixes_per_match = sixes/matches_stadium
sixes_per_match.sort_values(ascending=False)

In [ ]:
# Orange Cap Holders

ipl_batsman_runs_per_season = ipl.merge(matches,left_on="match_id",right_on="id").groupby(["season","batsman"])
ipl_batsman_runs_per_season["batsman_runs"].sum().reset_index().sort_values("batsman_runs",ascending=False).drop_duplicates("season",keep="first").sort_values("season")